In [4]:
# =========================
# 	 LOAD & ADD DOCUMENT
# =========================

import sys

sys.path.append("../")
from document_loader import document_loader
from vector_db import vector_db

collection_name = "semantic-chunk"

documents = document_loader(collection_name)
vector_db.add_document(
  collection_name = collection_name,
  documents = documents
)

Berhasil menambahkan 96 dokumen ke koleksi 'semantic-chunk'


In [ ]:
# ===============
# 	GET CONTEXT
# ===============

import sys
import pandas as pd
import time

sys.path.append("../")
from vector_db import vector_db

collection_name = "fixed-chunk"

data_test = pd.read_csv("../document/data_test_rag.csv", sep = ";")
retriever = vector_db.get_retriever(collection_name)

def get_context(question):
  t0 = time.perf_counter()
  docs = retriever.invoke(question)
  total_time = round(time.perf_counter() - t0, 2)

  context = "\n\n".join([doc.page_content for doc in docs])
  chunk_id = [doc.metadata.get("chunk_id") for doc in docs]

  return context, chunk_id, total_time

retrieved_context = []
chunk_id = []
duration = []

for question in data_test["question"]:
  context, id, t = get_context(question)
  
  retrieved_context.append(context)
  chunk_id.append(id)
  duration.append(t)

context_result = pd.DataFrame({
  f"{collection_name}_retrieved_context": retrieved_context,
  f"{collection_name}_id": chunk_id,
  f"{collection_name}_duration": duration
})

try:
  df = pd.read_csv(f"../result_data/result_data.csv", sep=";")
  df = pd.concat([df, context_result], axis=1)
except FileNotFoundError:
  df = context_result

df.to_csv(f"../result_data/result_data.csv", sep=";", index=False)

In [ ]:
# ===============
# 	GET ANSWER
# ===============

import sys
import pandas as pd

sys.path.append("../")
from rag import rag

BATCH = 1
BATCH_SIZE = 1
collection_name = "recursive-chunk"

start = (BATCH - 1) * BATCH_SIZE
end = BATCH * BATCH_SIZE

data_test = pd.read_csv("../document/data_test_rag.csv", sep = ";")
result_context = pd.read_csv(f"../result_data/result_data.csv", sep = ";")

questions = data_test["question"].tolist()
contexts = result_context[f"{collection_name}_retrieved_context"].to_list()

llm_response = []
duration_response = []

for question, context in zip(questions[start:end], contexts[start:end]):
  response, t = rag.get_answer(question, context)

  llm_response.append(response)
  duration_response.append(t)

rag_result = pd.DataFrame({
  f"{collection_name}_llm_response": llm_response,
  f"{collection_name}_duration_response": duration_response
})

try:
  df = pd.read_csv(f"../result_data/result_data", sep=";")
  df = pd.concat([df, rag_result], axis=1 ,ignore_index = True)

except FileNotFoundError:
  df = rag_result

df.to_csv(f"../result_data/result_data", sep=";", index=False)

In [ ]:
# ===============
# 	EVALUATION
# ===============

import sys
import types

dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [ ]:
from datasets import Dataset
import pandas as pd

collection_name = "fixed-chunk"

df_test = pd.read_csv(r"..\document\data_test_rag.csv", sep = ";")
df_result = pd.read_csv(r"..\result_data\result_data.csv", sep = ";")

df_eval = pd.concat([df_test.head(2), df_result.head(2)], axis=1)
df_eval = df_eval[
  [
    "question",
    f"{collection_name}_llm_response",
  ]
].copy()

df_eval.rename(
  columns={
    "question": "user_input",
    f"{collection_name}_llm_response": "response",
  },
  inplace=True,
)

dataset = Dataset.from_pandas(
  df_eval,
  preserve_index=False
)
dataset

c:\Users\reyha\VS Code\machine-learning\RAG AI\.ragvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['user_input', 'reference', 'retrieved_contexts', 'response'],
    num_rows: 2
})

In [ ]:
from ragas import evaluate
from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from langchain_ollama import OllamaEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
  answer_relevancy,
  context_precision,
  context_recall,
)

llm = ChatOllama( model="qwen3:8b", temperature=0 ) 
embedding = OllamaEmbeddings( model="bge-m3" )

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embedding = LangchainEmbeddingsWrapper(embedding)

run_config = RunConfig(
  timeout=600,
  max_workers=1,
  max_retries=3,
  max_wait=60,
)

result = evaluate(
  dataset=dataset,
  metrics=[
    answer_relevancy
  ],
  llm=evaluator_llm,
  embeddings=evaluator_embedding,
  run_config=run_config
)